# Use Cases Of An AI Engineer

Run `4-classify-job-postings.ipynb` first so that `jobs/2-llm_filtered_jobs.jsonl` exists.

This notebook reads the LLM-filtered AI engineer job postings and uses `gpt-5.4-mini` to summarize the common AI application use cases these companies are building.


In [1]:
import re
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import Markdown, display

llm_filtered_jsonl_path = Path("jobs") / "2-llm_filtered_jobs.jsonl"

if not llm_filtered_jsonl_path.exists():
    raise FileNotFoundError(
        f"LLM-filtered jobs JSONL file not found: {llm_filtered_jsonl_path.resolve()}. Run 4-classify-job-postings.ipynb first."
    )

jobs_for_summary = pd.read_json(llm_filtered_jsonl_path, lines=True)

if jobs_for_summary.empty:
    raise ValueError(
        "The LLM-filtered jobs JSONL file is empty. Run 4-classify-job-postings.ipynb first."
    )

if "description" not in jobs_for_summary.columns:
    raise KeyError(
        "The LLM-filtered jobs JSONL file must contain a 'description' column."
    )

jobs_for_summary = jobs_for_summary[jobs_for_summary["description"].notna()].copy()
jobs_for_summary["description"] = (
    jobs_for_summary["description"].astype(str).str.strip()
)
jobs_for_summary = jobs_for_summary[jobs_for_summary["description"] != ""].copy()

if "job_url" in jobs_for_summary.columns:
    jobs_for_summary = jobs_for_summary.drop_duplicates(subset=["job_url"])
else:
    jobs_for_summary = jobs_for_summary.drop_duplicates(
        subset=["title", "company", "description"]
    )

if jobs_for_summary.empty:
    raise ValueError(
        "No usable job descriptions remain after cleaning and deduplication."
    )


def clean_description(text):
    text = str(text)
    text = text.replace("**", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


max_description_chars = 10000
job_entries = []

for _, job in jobs_for_summary.iterrows():
    title = "" if pd.isna(job.get("title")) else str(job.get("title")).strip()
    company = "" if pd.isna(job.get("company")) else str(job.get("company")).strip()
    description = clean_description(job.get("description", ""))

    if not description:
        continue

    if len(description) > max_description_chars:
        description = (
            description[:max_description_chars]
            + " [Description truncated for summarization.]"
        )

    job_entries.append(
        "\n".join(
            [
                f"Title: {title}",
                f"Company: {company}",
                f"Description: {description}",
            ]
        )
    )

if not job_entries:
    raise ValueError("No non-empty job descriptions were available for summarization.")

job_descriptions = "\n\n---\n\n".join(job_entries)

client = OpenAI()

instructions = """
You analyze AI engineering job postings and identify the common application use cases the companies are building.

Focus on the actual product, workflow, or business use cases described across the postings.
Ignore company marketing, benefits, equal opportunity text, hiring process details, and generic recruiting language.
Do not focus on responsibilities or skills unless they directly clarify the application use case.
""".strip()

prompt = f"""
Read the job descriptions below and summarize the common AI application use cases these companies are building.

Return markdown in exactly this format:
## Shared use cases
- 5 to 10 concise bullet points

Requirements:
- Focus on the applications, workflows, or business use cases being built
- Keep bullets concrete and specific
- Do not include commentary before or after the bullets
- Do not mention company names
- Do not restate generic engineering responsibilities

Job descriptions:
{job_descriptions}
""".strip()

print(f"Summarizing use cases across {len(job_entries)} LLM-filtered job postings")
print(f"Combined description length: {len(job_descriptions):,} characters")

response = client.responses.create(
    model="gpt-5.4-mini",
    instructions=instructions,
    input=prompt,
    max_output_tokens=500,
    text={"verbosity": "low"},
)

summary = response.output_text
display(Markdown(summary))

Summarizing use cases across 40 LLM-filtered job postings
Combined description length: 213,455 characters


## Shared use cases
- Building agentic workflows that automate multi-step business tasks
- Creating internal AI assistants for knowledge lookup, decision support, and employee productivity
- Connecting LLMs to enterprise systems, APIs, databases, and documents for grounded responses
- Using RAG, embeddings, semantic search, and memory/context layers to retrieve relevant information
- Automating customer, employee, and operational support workflows with AI
- Producing natural-language interfaces over structured data, including text-to-SQL and conversational analytics
- Evaluating, monitoring, and governing AI outputs with test suites, observability, and human review
- Embedding AI into enterprise software for workflow orchestration, routing, and task execution
- Generating, summarizing, and classifying documents, messages, reports, and other business content
- Applying AI to domain-specific high-value workflows such as finance, HR, healthcare, ads, risk, security, and recruiting